Before running the notebook run the command `mlflow server --host 127.0.0.1 --port 5000`

In [5]:
from pathlib import Path

import mlflow
import pandas as pd
from hydra import compose, initialize
from pycaret import classification


# Load config

In [6]:
with initialize(version_base=None, config_path="../conf"):
    config = compose(config_name="health_prediction_config")

# Load data

In [7]:
health_df = pd.read_csv(Path("..") / config.silver_dataset_path)
health_df

,Age,Gender,Cholesterol,Glucose,Smoking,Alcohol Consumption,Exercise,BMI,Family History,Heart Disease,...,Stroke,Kidney Disease,Cancer,Alzheimer's Disease,COPD,Liver Disease,Parkinson's Disease,Tuberculosis,Blood Pressure_Low,Blood Pressure_Normal
0,69,0,1,1,1,0,0,35.671099,0,1,...,0,0,1,0,0,0,0,0,0,0
1,32,0,1,0,1,0,1,38.554188,1,0,...,0,0,0,0,0,1,0,0,1,0
2,89,1,1,0,0,0,1,18.932964,1,1,...,0,0,0,0,0,0,0,0,0,1
3,78,0,1,1,0,0,1,21.806350,1,0,...,1,0,1,0,0,1,0,0,0,0
4,38,0,0,0,1,1,1,37.552683,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,27,1,1,0,0,0,0,31.960176,1,1,...,0,0,0,0,0,0,0,0,1,0
996,51,1,1,0,0,1,1,20.118492,1,0,...,0,0,0,0,0,0,0,0,0,0
997,72,1,1,0,1,0,0,20.916536,1,0,...,0,0,0,0,0,0,0,0,0,1
998,49,0,1,1,1,0,1,19.560143,1,0,...,0,0,0,1,0,0,0,0,0,1


# Modelling

In [8]:
# Connect to Mlflow server
mlflow.set_tracking_uri("http://127.0.0.1:5000")


# Initialize PyCaret Setup
target_cols = ["Heart Disease", "Diabetes", "Stroke", "Kidney Disease", "Cancer", "Alzheimer's Disease", "COPD", "Liver Disease", "Parkinson's Disease", "Tuberculosis"]

reg_setups = {
    "with": {},
    "without": {},
}

for col in target_cols:
    reg_setups["with"][col] = classification.setup(
        data=health_df,
        target=col,
        train_size=config.train_size,
        fix_imbalance=True,
        transformation=True,
        normalize=True,
        session_id=config.random_state,
        log_experiment=True,
        experiment_name=f"health_{col.lower().replace(' ', '_').replace(',', ' ')}_with",
        memory=False,
    )

    reg_setups["without"][col] = classification.setup(
        data=health_df,
        target=col,
        train_size=config.train_size,
        ignore_features=["Age", "BMI"],
        fix_imbalance=True,
        transformation=True,
        normalize=True,
        session_id=config.random_state,
        log_experiment=True,
        experiment_name=f"health_{col.lower().replace(' ', '_').replace(',', ' ')}_without",
        memory=False,
    )


2026/08/21 00:11:48 INFO mlflow.tracking._tracking_service.client: 🏃 View run Session Initialized aea7 at: http://127.0.0.1:5000/#/experiments/190397121177740311/runs/c579be90df454eb8b7f98c7d43b8027b.
2026/08/21 00:11:48 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/190397121177740311.
2026/08/21 00:11:48 INFO mlflow.tracking._tracking_service.client: 🏃 View run Session Initialized 8371 at: http://127.0.0.1:5000/#/experiments/741725574466721952/runs/657e0742f514461786e853113aa494fc.
2026/08/21 00:11:48 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/741725574466721952.
2026/08/21 00:11:48 INFO mlflow.tracking._tracking_service.client: 🏃 View run Session Initialized 41a7 at: http://127.0.0.1:5000/#/experiments/455300831200369879/runs/8adf0679331643f2a480f5593a9395d4.
2026/08/21 00:11:48 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5

,Description,Value
0,Session id,67
1,Target,Heart Disease
2,Target type,Binary
3,Original data shape,"(1000, 21)"
4,Transformed data shape,"(1396, 21)"
5,Transformed train set shape,"(1196, 21)"
6,Transformed test set shape,"(200, 21)"
7,Numeric features,20
8,Preprocess,True
9,Imputation type,simple


MlflowException: Cannot set a deleted experiment 'health_heart_disease_with' as the active experiment. You can restore the experiment, or permanently delete the experiment to create a new one.